# Network Flow Encoder, MLP và Instance-wise Unlearning
Notebook này là file chính để chạy thí nghiệm theo plan: đọc raw network flow dạng PCAP, chuyển mỗi flow thành chuỗi packet features, encoder tạo vector đặc trưng 256 chiều, sau đó MLP phân loại nhị phân `known/unknown`.
Notebook chỉ hỗ trợ một mode chạy chính: Google Colab kết nối Google Drive tại `/content/drive`. Cấu hình dataset, nhãn `known/unknown`, holdout open-world và unlearning nằm ở cell cấu hình đầu notebook.


In [39]:
# OVERVIEW: Chuẩn bị dependency runtime cho Colab trước khi đọc PCAP và tạo DataLoader.
# Trong plan, input raw là các file .pcap; để parse PCAP cần Scapy. Cell này tự cài
# package còn thiếu để các cell sau có thể chạy theo thứ tự từ trên xuống.
import importlib.util
import subprocess
import sys

REQUIRED_IMPORTS = {
    'scapy': 'scapy',
    'googleapiclient': 'google-api-python-client',
}

missing_packages = [
    package_name
    for import_name, package_name in REQUIRED_IMPORTS.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print('Cài dependency còn thiếu:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Các dependency runtime đã sẵn sàng.')

for import_name in REQUIRED_IMPORTS:
    assert importlib.util.find_spec(import_name) is not None, f'Vẫn thiếu module {import_name}'


Các dependency runtime đã sẵn sàng.


In [40]:
# OVERVIEW: Cấu hình một mode duy nhất: Google Colab mount Google Drive, đọc PCAP baseline,
# ánh xạ nhãn gốc của dataset thành bài toán nhị phân known/unknown theo plan open-world.
from pathlib import Path

try:
    from google.colab import auth, drive
except ModuleNotFoundError as error:
    raise RuntimeError(
        'Notebook này được chuẩn hóa cho Google Colab. '
        'Hãy mở train_pipeline.ipynb trên Colab để có google.colab và /content/drive.'
    ) from error

# Folder gốc đã được xác nhận trong Colab. Tên có dấu cách cuối: "Traffic FingerPrinting ".
DRIVE_MOUNT_POINT = Path('/content/drive')
MY_DRIVE = DRIVE_MOUNT_POINT / 'MyDrive'
DRIVE_ROOT = MY_DRIVE / 'Traffic FingerPrinting '
DRIVE_ROOT_NAMES = ['Traffic FingerPrinting ', 'Traffic FingerPrinting', 'Tracffic FingerPrinting']
DRIVE_ROOT_FOLDER_ID = '10uCO2H7CPPS76wfxFnJWKtKM6HY9fGhJ'
DRIVE_OUTPUT_RELATIVE = Path('unlearning-artifacts/notebook')
REMOTE_CACHE_DIR = Path('/content/pcap_cache/traffic_fingerprinting')
FORCE_DRIVE_REMOUNT = False

# Các dataset đã kiểm tra. Không train tất cả ngay từ đầu để tránh trộn domain quá sớm.
CANDIDATE_DATA_DIR_LIST = [
    '/content/drive/MyDrive/Traffic FingerPrinting /Data/273 (200samples key)',
    '/content/drive/MyDrive/Traffic FingerPrinting /Data/273 (lan 1)',
    '/content/drive/MyDrive/Traffic FingerPrinting /Data/AOL (lan 1)',
    '/content/drive/MyDrive/Traffic FingerPrinting /Data/Data iPad/Data iPad new',
    '/content/drive/MyDrive/Traffic FingerPrinting /Data/Data iPhone/Data iPhone new',
]

# Baseline theo plan: train trước trên một dataset sạch, đã xác nhận 9005 PCAP.
# Khi cần đánh giá mở rộng/cross-domain, đổi dòng dưới thành CANDIDATE_DATA_DIR_LIST.
DATA_DIR_LIST = [CANDIDATE_DATA_DIR_LIST[0]]

DATASET_EVALUATION_NOTES = {
    '273 (200samples key)': 'Baseline tốt nhất: cấu trúc <label>/*.pcap, đã xác nhận 9005 file .pcap.',
    '273 (lan 1)': 'Cùng họ 273, phù hợp để mở rộng sau baseline và kiểm tra ổn định theo lần thu thập.',
    'AOL (lan 1)': 'Dùng được nhưng label là query phrase, nên ưu tiên cho open-world/generalization sau baseline.',
    'Data iPad/Data iPad new': 'Cross-device iPad, có tầng trung gian và PCAP lớn hơn; dùng sau khi pipeline ổn.',
    'Data iPhone/Data iPhone new': 'Cross-device iPhone, có tầng trung gian; phù hợp để test domain shift.',
}

# Relative path dùng cho fallback Drive API nếu shortcut/mounted path chưa khớp.
DRIVE_DATA_RELATIVE_LIST = [
    Path('Data') / '273 (200samples key)',
    Path('Data') / '273 (lan 1)',
    Path('Data') / 'AOL (lan 1)',
    Path('Data') / 'Data iPad' / 'Data iPad new',
    Path('Data') / 'Data iPhone' / 'Data iPhone new',
]

# Nhãn gốc có sẵn trong folder dataset. Ta chỉ ánh xạ chúng sang binary label:
# known = class mô hình được biết khi train; unknown-train = ví dụ unknown có trong train;
# holdout unknown = class không xuất hiện trong train, chỉ dùng để test open-world.
KNOWN_LABELS = {'english_to_spanish', 'office_365'}
UNKNOWN_LABELS = {'food_near_me'}
HOLDOUT_UNKNOWN_LABELS = {'twitter'}

# Hyperparameter chính của pipeline PCAP -> encoder 256 chiều -> MLP known/unknown.
MAX_PACKETS = 256
EMBEDDING_DIM = 256
HIDDEN_DIMS = (512, 256, 128, 64, 32)
DROPOUT = 0.20
BATCH_SIZE = 128
EPOCHS = 12
LEARNING_RATE = 5e-4
MAX_TRAIN_BATCHES_PER_EPOCH = None
MAX_EVAL_BATCHES = None
MAX_TEST_BATCHES = None
LOG_EVERY_N_BATCHES = 2
USE_CLASS_WEIGHTS = True
CHECKPOINT_SCORE_METRIC = 'balanced_accuracy'
UNKNOWN_THRESHOLD = 0.50
THRESHOLD_GRID = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
USE_THRESHOLD_SWEEP = True
SAVE_EMBEDDINGS_AFTER_TRAIN = False
MAX_EMBEDDING_RECORDS = 300
MAX_FILES_PER_LABEL = 160
SEED = 42
DEVICE_NAME = ''  # ''/'auto' = tự chọn GPU nếu khả dụng; đặt 'cuda' chỉ khi runtime thật sự có CUDA.
REQUIRE_CUDA = False  # True = báo lỗi nếu không có CUDA, không tự fallback CPU.

# Tự động chia nhãn ở cấp class để dùng nhiều nhãn hơn và tránh leakage giữa train/test.
# Mặc định tối ưu cho Colab free GPU: tăng unknown-train, dùng nhiều sample hơn và calibrate threshold.
# Khi cần baseline đầy đủ, đặt MAX_LABELS_FOR_EXPERIMENT=None và MAX_FILES_PER_LABEL=None.
AUTO_LABEL_SPLIT = True
REBUILD_LABEL_SPLIT = False
LABEL_SPLIT_SEED = SEED
MIN_SAMPLES_PER_LABEL = 100
KNOWN_LABEL_RATIO = 0.45
UNKNOWN_TRAIN_LABEL_RATIO = 0.35
HOLDOUT_VALIDATION_RATIO = 0.25
MAX_LABELS_FOR_EXPERIMENT = 24


def normalize_drive_name(name: str) -> str:
    """Normalize Drive folder names so trailing spaces or slash variants do not break matching."""
    return name.replace('/', ' ').strip().casefold()


def safe_cache_name(name: str) -> str:
    """Create a safe cache folder name for Drive API fallback downloads inside Colab."""
    return name.replace('/', '_').strip()


def resolve_drive_data_dirs() -> list[Path]:
    """Resolve active DATA_DIR_LIST from the mounted Drive filesystem first."""
    direct_dirs = [Path(path) for path in DATA_DIR_LIST]
    if all(path.is_dir() for path in direct_dirs):
        return direct_dirs

    missing_direct = [str(path) for path in direct_dirs if not path.is_dir()]
    print('Một số DATA_DIR_LIST chưa thấy qua mounted Drive:', missing_direct)

    active_relatives = [
        relative for relative in DRIVE_DATA_RELATIVE_LIST
        if str(DRIVE_ROOT / relative) in DATA_DIR_LIST
    ]
    if not active_relatives:
        active_relatives = DRIVE_DATA_RELATIVE_LIST[:len(DATA_DIR_LIST)]

    for root_name in DRIVE_ROOT_NAMES:
        root = MY_DRIVE / root_name
        source_dirs = [root / relative for relative in active_relatives]
        if root.is_dir() and all(path.is_dir() for path in source_dirs):
            return source_dirs

    raise FileNotFoundError(
        'Không tìm thấy đủ folder dữ liệu trong mounted Drive. Đã thử root: '
        + ', '.join(str(MY_DRIVE / name) for name in DRIVE_ROOT_NAMES)
    )


def download_drive_fallback(active_relatives: list[Path]) -> list[Path]:
    """Fallback khi mounted path không thấy dataset: dùng Drive API tải PCAP vào cache Colab."""
    try:
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
    except ModuleNotFoundError as error:
        raise RuntimeError(
            'Cần google-api-python-client để đọc folder shared. '
            'Hãy chạy lại cell dependency đầu notebook.'
        ) from error

    try:
        auth.authenticate_user()
    except Exception as error:
        raise RuntimeError(
            f'Xác thực Drive API thất bại ({type(error).__name__}: {error}). '
            'Nếu mounted Drive đã có đủ DATA_DIR_LIST thì không cần Drive API; '
            'nếu chưa có, hãy Add shortcut to Drive cho folder Traffic FingerPrinting hoặc reconnect runtime.'
        ) from error

    drive_api = build('drive', 'v3')

    def list_drive_children(folder_id: str) -> list[dict[str, str]]:
        files = []
        page_token = None
        while True:
            response = drive_api.files().list(
                q=f"'{folder_id}' in parents and trashed = false",
                spaces='drive',
                fields='nextPageToken, files(id, name, mimeType)',
                pageSize=1000,
                pageToken=page_token,
                includeItemsFromAllDrives=True,
                supportsAllDrives=True,
            ).execute()
            files.extend(response.get('files', []))
            page_token = response.get('nextPageToken')
            if not page_token:
                return files

    def find_child(folder_id: str, name: str) -> dict[str, str]:
        children = list_drive_children(folder_id)
        target_name = normalize_drive_name(name)
        matches = [
            item for item in children
            if item['name'] == name or normalize_drive_name(item['name']) == target_name
        ]
        if len(matches) != 1:
            available = sorted(item['name'] for item in children)[:50]
            raise FileNotFoundError(
                f'Không tìm thấy duy nhất folder {name!r} trong Drive API. Folder hiện có: {available}'
            )
        return matches[0]

    def find_relative_folder(root_folder_id: str, relative_path: Path) -> dict[str, str]:
        current = {'id': root_folder_id, 'name': 'root'}
        for part in relative_path.parts:
            current = find_child(current['id'], part)
        return current

    def safe_relative_name(relative_path: Path) -> str:
        return '__'.join(safe_cache_name(part) for part in relative_path.parts if part != 'Data')

    def download_tree(folder_id: str, target_dir: Path) -> None:
        target_dir.mkdir(parents=True, exist_ok=True)
        for item in list_drive_children(folder_id):
            if item.get('mimeType') == 'application/vnd.google-apps.folder':
                download_tree(item['id'], target_dir / safe_cache_name(item['name']))
                continue
            if not item.get('name', '').lower().endswith(('.pcap', '.cap')):
                continue
            target = target_dir / safe_cache_name(item['name'])
            if target.exists():
                continue
            request = drive_api.files().get_media(fileId=item['id'])
            with target.open('wb') as handle:
                downloader = MediaIoBaseDownload(handle, request)
                done = False
                while not done:
                    _, done = downloader.next_chunk()
            print(f'Tải {target}')

    REMOTE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for relative in active_relatives:
        source_folder = find_relative_folder(DRIVE_ROOT_FOLDER_ID, relative)
        download_tree(source_folder['id'], REMOTE_CACHE_DIR / safe_relative_name(relative))
    return [REMOTE_CACHE_DIR / safe_relative_name(relative) for relative in active_relatives]


drive.mount(str(DRIVE_MOUNT_POINT), force_remount=FORCE_DRIVE_REMOUNT)
if not MY_DRIVE.is_dir():
    raise RuntimeError(f'Mount xong nhưng không thấy {MY_DRIVE}. Kiểm tra tài khoản Google Drive đang dùng trong Colab.')

try:
    DATA_DIR = resolve_drive_data_dirs()
except FileNotFoundError as path_error:
    print(path_error)
    print('Chuyển sang Drive API và tải đệ quy các file PCAP vào cache.')
    active_relative_paths = [Path(path).relative_to(DRIVE_ROOT) for path in map(Path, DATA_DIR_LIST)]
    DATA_DIR = download_drive_fallback(active_relative_paths)

OUTPUT_DIR = MY_DRIVE / DRIVE_OUTPUT_RELATIVE
if not all(path.exists() and path.is_dir() for path in DATA_DIR):
    raise FileNotFoundError(
        f'Không tìm thấy DATA_DIR: {DATA_DIR}\n'
        'Kiểm tra shortcut Drive, DATA_DIR_LIST và quyền truy cập folder.'
    )

print('Runtime: Google Colab + Google Drive')
print('DATA_DIR đang train:')
for path in DATA_DIR:
    print(f'  - {path}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print('Dataset notes:')
for name, note in DATASET_EVALUATION_NOTES.items():
    print(f'  - {name}: {note}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: Google Colab + Google Drive
DATA_DIR đang train:
  - /content/drive/MyDrive/Traffic FingerPrinting /Data/273 (200samples key)
OUTPUT_DIR: /content/drive/MyDrive/unlearning-artifacts/notebook
Dataset notes:
  - 273 (200samples key): Baseline tốt nhất: cấu trúc <label>/*.pcap, đã xác nhận 9005 file .pcap.
  - 273 (lan 1): Cùng họ 273, phù hợp để mở rộng sau baseline và kiểm tra ổn định theo lần thu thập.
  - AOL (lan 1): Dùng được nhưng label là query phrase, nên ưu tiên cho open-world/generalization sau baseline.
  - Data iPad/Data iPad new: Cross-device iPad, có tầng trung gian và PCAP lớn hơn; dùng sau khi pipeline ổn.
  - Data iPhone/Data iPhone new: Cross-device iPhone, có tầng trung gian; phù hợp để test domain shift.


In [41]:
# OVERVIEW: Nạp thư viện dùng chung, kiểm tra CUDA và chọn DEVICE CPU/GPU an toàn cho pipeline Colab.
from __future__ import annotations

import argparse
import hashlib
import copy
import csv
import json
import math
import random
import time
from collections import Counter
from dataclasses import dataclass
from itertools import cycle
from pathlib import Path
from typing import Sequence

import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset


PACKET_FEATURES = 5
NUM_CLASSES = 2


def select_device(device_name: str = '', require_cuda: bool = False) -> torch.device:
    """Chọn device an toàn: chỉ dùng cuda khi PyTorch/runtime thật sự hỗ trợ CUDA."""
    requested = (device_name or '').strip().lower()
    if requested == 'gpu':
        requested = 'cuda'
    if requested not in {'', 'auto', 'cpu', 'cuda'}:
        raise ValueError("DEVICE_NAME chỉ nên là '', 'auto', 'cpu' hoặc 'cuda'.")

    cuda_compiled = hasattr(torch._C, '_cuda_getDeviceCount')
    cuda_available = bool(cuda_compiled and torch.cuda.is_available())
    print(f'PyTorch version: {torch.__version__}')
    print(f'torch_cuda_compiled={cuda_compiled}, torch_cuda_available={cuda_available}')

    if requested == 'cpu':
        return torch.device('cpu')

    if requested in {'', 'auto'}:
        if cuda_available:
            device = torch.device('cuda')
            print(f'CUDA device: {torch.cuda.get_device_name(0)}')
            return device
        message = 'CUDA không khả dụng, fallback sang CPU.'
        if require_cuda:
            raise RuntimeError(
                message + ' Hãy vào Colab: Runtime > Change runtime type > Hardware accelerator > GPU, '
                'sau đó restart session và chạy lại từ cell dependency/import.'
            )
        print(message)
        return torch.device('cpu')

    if requested == 'cuda' and not cuda_available:
        raise RuntimeError(
            "DEVICE_NAME='cuda' nhưng runtime PyTorch hiện tại không dùng được CUDA. "
            f"torch_cuda_compiled={cuda_compiled}, torch_cuda_available={cuda_available}. "
            "Nếu đang ở Colab, vào Runtime > Change runtime type > Hardware accelerator > GPU, "
            "restart session, rồi chạy lại từ cell dependency/import. Nếu đang chạy kernel local/VS Code CPU-only, "
            "đặt DEVICE_NAME='' hoặc 'cpu'."
        )

    device = torch.device('cuda')
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    return device


DEVICE = select_device(DEVICE_NAME, REQUIRE_CUDA)
print(f'Device đang dùng: {DEVICE}')


PyTorch version: 2.11.0+cu128
torch_cuda_compiled=True, torch_cuda_available=True
CUDA device: Tesla T4
Device đang dùng: cuda


In [42]:
# OVERVIEW: Mỗi FlowRecord lưu path PCAP, nhãn gốc từ folder, và binary label known=0/unknown=1.
@dataclass(frozen=True)
class FlowRecord:
    path: Path
    original_label: str
    binary_label: int


In [43]:
# OVERVIEW: Cố định seed để split train/validation/test và quá trình train có thể lặp lại.
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [44]:
# OVERVIEW: Helper đọc danh sách nhãn dạng CSV, dùng cho CLI/phần tái sử dụng ngoài notebook.
def parse_csv_list(value: str | None) -> set[str]:
    if not value:
        return set()
    return {item.strip() for item in value.split(",") if item.strip()}


In [45]:
# OVERVIEW: Suy luận nhãn gốc từ folder cha trực tiếp của PCAP, vì dataset đã có nhãn theo thư mục.
def infer_label(path: Path, root: Path) -> str:
    """Infer a class label from a label directory or a traffic_*.txt name."""
    if path.parent.resolve() != root.resolve():
        return path.parent.name

    stem = path.stem
    parts = stem.split("_")
    if parts and parts[-1].isdigit():
        parts = parts[:-1]
    label = "_".join(parts)
    if label.startswith("traffic_"):
        label = label[len("traffic_"):]
    return label or stem


In [46]:
# OVERVIEW: Quét đệ quy DATA_DIR để lấy các raw flow .pcap/.cap; mỗi file là một sample theo plan.
def scan_flow_files(root: Path | Sequence[Path]) -> list[tuple[Path, str]]:
    roots = [root] if isinstance(root, Path) else list(root)
    if not roots:
        raise ValueError("Cần ít nhất một folder dữ liệu")
    extensions = {".txt", ".csv", ".pcap", ".cap"}
    files: list[tuple[Path, str]] = []
    for source_root in roots:
        if not source_root.exists() or not source_root.is_dir():
            raise FileNotFoundError(f"Không tìm thấy folder dữ liệu: {source_root}")
        files.extend(
            (path, infer_label(path, source_root))
            for path in source_root.rglob("*")
            if path.is_file() and path.suffix.lower() in extensions
        )
    files.sort()
    if not files:
        raise FileNotFoundError(
            f"Không tìm thấy file .txt/.csv/.pcap/.cap trong {[str(path) for path in roots]}"
        )
    return files


In [47]:
# OVERVIEW: Đọc packet IPv4/IPv6 từ PCAP để lấy timestamp, IP, port và kích thước packet thô.
def read_pcap_packets(path: Path) -> list[tuple[float, str, int, str, int, float]]:
    """Read IPv4/IPv6 packets from a pcap and expose the CSV-like fields."""
    try:
        from scapy.all import PcapReader
        from scapy.layers.inet import IP, TCP, UDP
        from scapy.layers.inet6 import IPv6
    except ImportError as error:
        raise RuntimeError(
            "Đọc .pcap cần Scapy. Hãy chạy cell dependency đầu notebook, "
            "hoặc cài thủ công bằng: pip install scapy"
        ) from error

    packets: list[tuple[float, str, int, str, int, float]] = []
    with PcapReader(str(path)) as capture:
        for packet in capture:
            ip_layer = packet.getlayer(IP)
            if ip_layer is None:
                ip_layer = packet.getlayer(IPv6)
            if ip_layer is None:
                continue

            transport_layer = packet.getlayer(TCP)
            if transport_layer is None:
                transport_layer = packet.getlayer(UDP)

            src_port = int(getattr(transport_layer, "sport", 0) or 0)
            dst_port = int(getattr(transport_layer, "dport", 0) or 0)
            timestamp = float(packet.time)
            packets.append(
                (
                    timestamp,
                    str(ip_layer.src),
                    max(0, min(src_port, 65535)),
                    str(ip_layer.dst),
                    max(0, min(dst_port, 65535)),
                    float(len(packet)),
                )
            )
    return packets


In [48]:
# OVERVIEW: Chuyển raw flow thành tensor [MAX_PACKETS, 5] và mask; đây là input trực tiếp của encoder.
def parse_flow_file(path: Path, max_packets: int) -> tuple[Tensor, Tensor]:
    """Convert one packet-flow file into (features, valid_mask).

    Feature order per packet:
        log(1 + relative_time), direction, source_port, destination_port,
        log(1 + packet_size)
    """
    packets: list[tuple[float, str, int, str, int, float]] = []
    if path.suffix.lower() in {".pcap", ".cap"}:
        packets = read_pcap_packets(path)
    else:
        with path.open("r", encoding="utf-8", errors="ignore", newline="") as handle:
            for row in csv.reader(handle):
                if len(row) < 6:
                    continue
                try:
                    timestamp = float(row[0].strip())
                    src_ip = row[1].strip()
                    src_port = int(float(row[2].strip()))
                    dst_ip = row[3].strip()
                    dst_port = int(float(row[4].strip()))
                    packet_size = float(row[5].strip())
                except (ValueError, TypeError):
                    # Skips headers and malformed packet rows.
                    continue
                if not src_ip or not dst_ip:
                    continue
                packets.append(
                    (
                        timestamp,
                        src_ip,
                        max(0, min(src_port, 65535)),
                        dst_ip,
                        max(0, min(dst_port, 65535)),
                        max(0.0, packet_size),
                    )
                )

    features = torch.zeros(max_packets, PACKET_FEATURES, dtype=torch.float32)
    mask = torch.zeros(max_packets, dtype=torch.float32)
    if not packets:
        return features, mask

    local_ip = Counter(
        ip for _, src_ip, _, dst_ip, _, _ in packets for ip in (src_ip, dst_ip)
    ).most_common(1)[0][0]
    start_time = packets[0][0]

    for index, (timestamp, src_ip, src_port, _, dst_port, packet_size) in enumerate(
        packets[:max_packets]
    ):
        relative_time = max(0.0, timestamp - start_time)
        direction = 0.0 if src_ip == local_ip else 1.0
        features[index] = torch.tensor(
            [
                math.log1p(relative_time),
                direction,
                src_port / 65535.0,
                dst_port / 65535.0,
                math.log1p(packet_size) / math.log1p(65535.0),
            ],
            dtype=torch.float32,
        )
        mask[index] = 1.0

    return features, mask


In [49]:
# OVERVIEW: Ánh xạ nhãn gốc sang known/unknown; không gán ngẫu nhiên nhãn, chỉ dùng cấu hình trong cell đầu.
def build_records(
    root: Path | Sequence[Path],
    known_labels: set[str],
    unknown_labels: set[str],
    holdout_unknown_labels: set[str],
) -> list[FlowRecord]:
    if not known_labels:
        raise ValueError("Cần chỉ định ít nhất một nhãn trong --known-labels")

    overlap = known_labels & (unknown_labels | holdout_unknown_labels)
    if overlap:
        raise ValueError(f"Nhãn xuất hiện ở nhiều nhóm: {sorted(overlap)}")

    files = scan_flow_files(root)
    discovered_labels = {label for _, label in files}
    unknown_labels = set(unknown_labels)
    if not unknown_labels:
        unknown_labels = discovered_labels - known_labels - holdout_unknown_labels
        print(
            "Cảnh báo: --unknown-labels chưa được chỉ định; "
            "các nhãn còn lại sẽ được dùng làm unknown train."
        )

    allowed = known_labels | unknown_labels | holdout_unknown_labels
    records: list[FlowRecord] = []
    for path, label in files:
        if label not in allowed:
            continue
        binary_label = 0 if label in known_labels else 1
        records.append(FlowRecord(path, label, binary_label))

    if not records:
        raise ValueError(
            "Không có flow nào khớp với known/unknown labels. "
            f"Labels tìm thấy: {sorted(discovered_labels)}"
        )
    return records


In [50]:
# OVERVIEW: Chia train/validation/test; holdout unknown không vào train nhưng một phần vào validation để calibrate threshold open-world.
def split_records(
    records: Sequence[FlowRecord],
    holdout_unknown_labels: set[str],
    seed: int,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    holdout_validation_ratio: float = 0.25,
) -> tuple[list[FlowRecord], list[FlowRecord], list[FlowRecord]]:
    """Split records; held-out unknown labels are excluded from train and split into validation/test."""
    rng = random.Random(seed)
    train: list[FlowRecord] = []
    validation: list[FlowRecord] = []
    test: list[FlowRecord] = []
    holdout_unknown: list[FlowRecord] = []

    grouped: dict[int, list[FlowRecord]] = {0: [], 1: []}
    for record in records:
        if record.original_label in holdout_unknown_labels:
            holdout_unknown.append(record)
        else:
            grouped[record.binary_label].append(record)

    for label, group in grouped.items():
        rng.shuffle(group)
        n = len(group)
        n_train = max(1, int(n * train_ratio)) if n >= 3 else max(0, n - 1)
        n_val = max(1, int(n * val_ratio)) if n >= 5 else 0
        if n_train + n_val >= n:
            n_val = max(0, n - n_train - 1)
        train.extend(group[:n_train])
        validation.extend(group[n_train:n_train + n_val])
        test.extend(group[n_train + n_val:])

    rng.shuffle(holdout_unknown)
    if holdout_unknown:
        n_holdout_val = int(len(holdout_unknown) * holdout_validation_ratio)
        n_holdout_val = min(max(1, n_holdout_val), max(len(holdout_unknown) - 1, 1))
        validation.extend(holdout_unknown[:n_holdout_val])
        test.extend(holdout_unknown[n_holdout_val:])

    rng.shuffle(train)
    rng.shuffle(validation)
    rng.shuffle(test)

    if not train or len({record.binary_label for record in train}) < NUM_CLASSES:
        raise ValueError(
            "Tập train phải có cả known và unknown. "
            "Hãy chỉ định UNKNOWN_LABELS có dữ liệu train."
        )
    return train, validation, test


In [51]:
# OVERVIEW: Dataset PyTorch parse PCAP theo kiểu lazy và cache tensor để cell tạo DataLoader không bị treo.
class FlowDataset(Dataset[tuple[Tensor, Tensor, Tensor]]):
    def __init__(self, records: Sequence[FlowRecord], max_packets: int, cache_dir: Path | None = None):
        self.records = list(records)
        self.max_packets = max_packets
        self.cache_dir = cache_dir
        if self.cache_dir is not None:
            self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.labels = torch.tensor(
            [record.binary_label for record in self.records], dtype=torch.long
        )

    def cache_path(self, record: FlowRecord) -> Path | None:
        if self.cache_dir is None:
            return None
        digest = hashlib.sha1(str(record.path).encode('utf-8')).hexdigest()[:16]
        safe_label = record.original_label.replace('/', '_').replace(' ', '_')
        return self.cache_dir / f'{safe_label}_{digest}.pt'

    def load_or_parse(self, record: FlowRecord) -> tuple[Tensor, Tensor]:
        cache_path = self.cache_path(record)
        if cache_path is not None and cache_path.exists():
            cached = torch.load(cache_path, map_location='cpu')
            return cached['features'], cached['mask']

        features, mask = parse_flow_file(record.path, self.max_packets)
        if cache_path is not None:
            torch.save({'features': features, 'mask': mask}, cache_path)
        return features, mask

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor]:
        features, mask = self.load_or_parse(self.records[index])
        return features, mask, self.labels[index]


In [52]:
# OVERVIEW: Encoder CNN 1D nén chuỗi packet thành vector đặc trưng 256 chiều như mô tả trong plan.
class FlowEncoder(nn.Module):
    """Encode a padded packet sequence into a 256-dimensional flow vector."""

    def __init__(self, embedding_dim: int = 256):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(PACKET_FEATURES, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )
        self.projection = nn.Sequential(
            nn.Linear(128, embedding_dim),
            nn.LayerNorm(embedding_dim),
        )

    def forward(self, features: Tensor, mask: Tensor) -> Tensor:
        # features: [batch, packets, packet_features]
        hidden = self.network(features.transpose(1, 2)).transpose(1, 2)
        valid = mask.unsqueeze(-1)
        pooled = (hidden * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1.0)
        return self.projection(pooled)


In [53]:
# OVERVIEW: MLP 5 hidden layers nhận embedding 256 chiều và xuất 2 logits: known và unknown.
class MLPClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int = 256,
        hidden_dims: Sequence[int] = (512, 256, 128, 64, 32),
        dropout: float = 0.20,
    ):
        super().__init__()
        layers: list[nn.Module] = []
        current_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend(
                [
                    nn.Linear(current_dim, hidden_dim),
                    nn.LayerNorm(hidden_dim),
                    nn.GELU(),
                    nn.Dropout(dropout),
                ]
            )
            current_dim = hidden_dim
        layers.append(nn.Linear(current_dim, NUM_CLASSES))
        self.network = nn.Sequential(*layers)

    def forward(self, embedding: Tensor) -> Tensor:
        return self.network(embedding)


In [54]:
# OVERVIEW: FlowModel ghép encoder và MLP thành pipeline end-to-end PCAP -> embedding -> known/unknown.
class FlowModel(nn.Module):
    def __init__(
        self,
        embedding_dim: int = 256,
        hidden_dims: Sequence[int] = (512, 256, 128, 64, 32),
        dropout: float = 0.20,
    ):
        super().__init__()
        self.encoder = FlowEncoder(embedding_dim)
        self.classifier = MLPClassifier(embedding_dim, hidden_dims, dropout)

    def forward(self, features: Tensor, mask: Tensor) -> tuple[Tensor, Tensor]:
        embedding = self.encoder(features, mask)
        logits = self.classifier(embedding)
        return logits, embedding


In [55]:
# OVERVIEW: Tạo DataLoader lazy; PCAP chưa parse hàng loạt tại bước này mà parse khi batch được đọc.
def make_loader(
    records: Sequence[FlowRecord],
    max_packets: int,
    batch_size: int,
    shuffle: bool,
    cache_dir: Path | None = None,
) -> DataLoader:
    return DataLoader(
        FlowDataset(records, max_packets, cache_dir=cache_dir),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


In [56]:
# OVERVIEW: Đưa batch packet tensor, mask và label lên CPU/GPU đã chọn trong cell cấu hình.
def move_batch(
    batch: tuple[Tensor, Tensor, Tensor], device: torch.device
) -> tuple[Tensor, Tensor, Tensor]:
    features, mask, labels = batch
    return features.to(device), mask.to(device), labels.to(device)


In [57]:
# OVERVIEW: Đánh giá loss, accuracy, balanced accuracy, recall từng lớp, threshold và confusion matrix cho known/unknown.
def evaluate(
    model: FlowModel,
    loader: DataLoader,
    device: torch.device,
    max_batches: int | None = None,
    unknown_threshold: float = 0.50,
) -> dict[str, object]:
    if len(loader.dataset) == 0:
        return {
            "loss": float("nan"),
            "accuracy": float("nan"),
            "known_recall": float("nan"),
            "unknown_recall": float("nan"),
            "unknown_precision": float("nan"),
            "balanced_accuracy": float("nan"),
            "unknown_threshold": unknown_threshold,
            "confusion_matrix": [[0, 0], [0, 0]],
        }

    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    total = 0
    correct = 0
    known_total = 0
    known_correct = 0
    unknown_total = 0
    unknown_correct = 0
    predicted_unknown_total = 0
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)

    with torch.no_grad():
        for batch_index, batch in enumerate(loader, start=1):
            if max_batches is not None and batch_index > max_batches:
                break
            features, mask, labels = move_batch(batch, device)
            logits, _ = model(features, mask)
            total_loss += criterion(logits, labels).item() * labels.numel()
            probabilities = torch.softmax(logits, dim=1)
            predictions = (probabilities[:, 1] >= unknown_threshold).long()
            correct += (predictions == labels).sum().item()
            total += labels.numel()

            known = labels == 0
            unknown = labels == 1
            predicted_unknown = predictions == 1
            known_total += known.sum().item()
            known_correct += ((predictions == labels) & known).sum().item()
            unknown_total += unknown.sum().item()
            unknown_correct += ((predictions == labels) & unknown).sum().item()
            predicted_unknown_total += predicted_unknown.sum().item()

            for true_label, predicted_label in zip(labels.detach().cpu(), predictions.detach().cpu()):
                confusion[int(true_label), int(predicted_label)] += 1

    accuracy = correct / max(total, 1)
    known_recall = known_correct / max(known_total, 1)
    unknown_recall = unknown_correct / max(unknown_total, 1)
    unknown_precision = unknown_correct / max(predicted_unknown_total, 1)
    balanced_accuracy = 0.5 * (known_recall + unknown_recall)

    return {
        "loss": total_loss / max(total, 1),
        "accuracy": accuracy,
        "known_recall": known_recall,
        "unknown_recall": unknown_recall,
        "unknown_precision": unknown_precision,
        "balanced_accuracy": balanced_accuracy,
        "unknown_threshold": unknown_threshold,
        "known_total": known_total,
        "unknown_total": unknown_total,
        "predicted_unknown_total": predicted_unknown_total,
        "confusion_matrix": confusion.tolist(),
    }


In [58]:
# OVERVIEW: Huấn luyện encoder + MLP bằng weighted CrossEntropyLoss và chọn checkpoint theo metric cân bằng cho open-world.
def build_class_weights(loader: DataLoader, device: torch.device, enabled: bool) -> Tensor | None:
    if not enabled or not hasattr(loader.dataset, 'labels'):
        return None
    labels = loader.dataset.labels.detach().cpu()
    counts = torch.bincount(labels, minlength=NUM_CLASSES).float()
    if (counts == 0).any():
        print(f'Không dùng class weights vì thiếu class trong train counts={counts.tolist()}')
        return None
    weights = counts.sum() / (NUM_CLASSES * counts)
    weights = weights / weights.mean()
    print(f'Train label counts={counts.int().tolist()}, class_weights={weights.tolist()}')
    return weights.to(device)


def metric_value(metrics: dict[str, object], name: str) -> float:
    if name == 'open_world_score':
        accuracy = float(metrics.get('accuracy', float('nan')))
        unknown_recall = float(metrics.get('unknown_recall', float('nan')))
        return 0.5 * accuracy + 0.5 * unknown_recall
    value = metrics.get(name, float('nan'))
    return float(value) if isinstance(value, (int, float)) else float('nan')


def train_model(
    model: FlowModel,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int,
    learning_rate: float,
    max_train_batches_per_epoch: int | None = None,
    max_eval_batches: int | None = None,
    log_every_n_batches: int = 5,
    use_class_weights: bool = True,
    checkpoint_score_metric: str = 'balanced_accuracy',
    unknown_threshold: float = 0.50,
) -> dict[str, object]:
    model.to(device)
    class_weights = build_class_weights(train_loader, device, use_class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_state = copy.deepcopy(model.state_dict())
    best_score = -float("inf")
    history: list[dict[str, float]] = []
    run_start = time.perf_counter()

    print(
        f'Train config: epochs={epochs}, batch_size={train_loader.batch_size}, '
        f'train_batches={len(train_loader)}, val_batches={len(val_loader)}, '
        f'max_train_batches_per_epoch={max_train_batches_per_epoch}, max_eval_batches={max_eval_batches}, '
        f'log_every_n_batches={log_every_n_batches}, use_class_weights={use_class_weights}, '
        f'checkpoint_score_metric={checkpoint_score_metric}, unknown_threshold={unknown_threshold}'
    )

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_start = time.perf_counter()
        total_loss = 0.0
        total = 0
        batches_seen = 0
        for batch_index, batch in enumerate(train_loader, start=1):
            if max_train_batches_per_epoch is not None and batch_index > max_train_batches_per_epoch:
                break
            features, mask, labels = move_batch(batch, device)
            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(features, mask)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item() * labels.numel()
            total += labels.numel()
            batches_seen += 1

            should_log = batch_index == 1 or batch_index % max(log_every_n_batches, 1) == 0
            if should_log:
                elapsed = time.perf_counter() - epoch_start
                print(
                    f'Epoch {epoch:03d}/{epochs} | batch {batch_index:04d}/{len(train_loader)} | '
                    f'loss={loss.item():.4f} | elapsed={elapsed:.1f}s'
                )

        val_start = time.perf_counter()
        val_metrics = evaluate(model, val_loader, device, max_batches=max_eval_batches, unknown_threshold=unknown_threshold)
        val_seconds = time.perf_counter() - val_start
        epoch_seconds = time.perf_counter() - epoch_start
        score = metric_value(val_metrics, checkpoint_score_metric)
        if math.isnan(score):
            score = -float("inf")
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
        row = {
            "epoch": float(epoch),
            "train_loss": total_loss / max(total, 1),
            "train_batches": float(batches_seen),
            "val_loss": float(val_metrics["loss"]),
            "val_accuracy": float(val_metrics["accuracy"]),
            "val_known_recall": float(val_metrics["known_recall"]),
            "val_unknown_recall": float(val_metrics["unknown_recall"]),
            "val_balanced_accuracy": float(val_metrics["balanced_accuracy"]),
            "checkpoint_score": score,
            "epoch_seconds": epoch_seconds,
            "val_seconds": val_seconds,
        }
        history.append(row)
        print(
            f"Epoch {epoch:03d}/{epochs} done | "
            f"train_loss={row['train_loss']:.4f} | "
            f"val_acc={row['val_accuracy']:.4f} | "
            f"val_known_recall={row['val_known_recall']:.4f} | "
            f"val_unknown_recall={row['val_unknown_recall']:.4f} | "
            f"val_balanced_acc={row['val_balanced_accuracy']:.4f} | "
            f"score={row['checkpoint_score']:.4f} | "
            f"epoch_time={epoch_seconds:.1f}s | val_time={val_seconds:.1f}s | "
            f"confusion={val_metrics['confusion_matrix']}"
        )

    model.load_state_dict(best_state)
    total_seconds = time.perf_counter() - run_start
    print(f'Train finished in {total_seconds:.1f}s')
    return {"history": history, "best_score": best_score, "total_seconds": total_seconds}


In [59]:
# OVERVIEW: Ước lượng độ quan trọng tham số trên retain set, dùng cho instance-wise unlearning.
def estimate_weight_importance(
    model: FlowModel,
    retain_loader: DataLoader,
    device: torch.device,
    max_batches: int = 100,
) -> dict[str, Tensor]:
    """Estimate diagonal Fisher-style importance from retain samples."""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    importance = {
        name: torch.zeros_like(parameter, device=device)
        for name, parameter in model.named_parameters()
    }
    for batch_index, batch in enumerate(retain_loader):
        if batch_index >= max_batches:
            break
        features, mask, labels = move_batch(batch, device)
        model.zero_grad(set_to_none=True)
        logits, _ = model(features, mask)
        criterion(logits, labels).backward()
        for name, parameter in model.named_parameters():
            if parameter.grad is not None:
                importance[name] += parameter.grad.detach().pow(2)

    for name, values in importance.items():
        mean = values.mean().clamp_min(1e-12)
        importance[name] = values / mean
    model.zero_grad(set_to_none=True)
    return importance


In [60]:
# OVERVIEW: Regularization giữ tham số quan trọng gần model gốc để bảo toàn Dr khi unlearning.
def parameter_anchor_loss(
    model: FlowModel,
    reference_state: dict[str, Tensor],
    importance: dict[str, Tensor],
) -> Tensor:
    loss = torch.zeros((), device=next(model.parameters()).device)
    for name, parameter in model.named_parameters():
        loss = loss + (importance[name] * (parameter - reference_state[name]).pow(2)).mean()
    return loss


In [61]:
# OVERVIEW: Instance-wise unlearning: flip target của Df, đồng thời giữ hiệu năng trên Dr bằng retain loss.
def run_unlearning(
    model: FlowModel,
    retain_loader: DataLoader,
    forget_loader: DataLoader,
    device: torch.device,
    epochs: int,
    learning_rate: float,
    retain_weight: float,
    regularization_weight: float,
) -> list[dict[str, float]]:
    """Perform targeted instance-wise relabeling with retain regularization."""
    model.to(device)
    model.eval()
    reference_state = {
        name: parameter.detach().clone()
        for name, parameter in model.named_parameters()
    }
    importance = estimate_weight_importance(model, retain_loader, device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    steps = max(len(retain_loader), len(forget_loader))
    retain_batches = cycle(retain_loader)
    forget_batches = cycle(forget_loader)
    history: list[dict[str, float]] = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for _ in range(steps):
            retain_features, retain_mask, retain_labels = move_batch(next(retain_batches), device)
            forget_features, forget_mask, forget_labels = move_batch(next(forget_batches), device)
            optimizer.zero_grad(set_to_none=True)

            retain_logits, _ = model(retain_features, retain_mask)
            forget_logits, _ = model(forget_features, forget_mask)
            forget_target = 1 - forget_labels
            forget_loss = criterion(forget_logits, forget_target)
            retain_loss = criterion(retain_logits, retain_labels)
            reg_loss = parameter_anchor_loss(model, reference_state, importance)
            loss = forget_loss + retain_weight * retain_loss + regularization_weight * reg_loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item()

        row = {"epoch": float(epoch), "loss": total_loss / max(steps, 1)}
        history.append(row)
        print(f"Unlearning epoch {epoch:03d}/{epochs} | loss={row['loss']:.6f}")
    return history


In [62]:
# OVERVIEW: Trích xuất embedding 256 chiều; dùng cache để tránh parse lại PCAP khi lưu feature vector.
def extract_embeddings(
    model: FlowModel,
    records: Sequence[FlowRecord],
    max_packets: int,
    batch_size: int,
    device: torch.device,
    cache_dir: Path | None = None,
) -> Tensor:
    loader = make_loader(records, max_packets, batch_size, shuffle=False, cache_dir=cache_dir)
    model.eval()
    embeddings: list[Tensor] = []
    with torch.no_grad():
        for batch in loader:
            features, mask, _ = move_batch(batch, device)
            _, batch_embeddings = model(features, mask)
            embeddings.append(batch_embeddings.cpu())
    return torch.cat(embeddings, dim=0) if embeddings else torch.empty(0, 256)


In [63]:
# OVERVIEW: Lưu embedding, nhãn known/unknown, nhãn gốc và path PCAP; dùng lại cache packet features.
def save_embeddings(
    model: FlowModel,
    records: Sequence[FlowRecord],
    output_path: Path,
    max_packets: int,
    batch_size: int,
    device: torch.device,
    cache_dir: Path | None = None,
) -> None:
    embeddings = extract_embeddings(model, records, max_packets, batch_size, device, cache_dir=cache_dir)
    torch.save(
        {
            "embeddings": embeddings,
            "binary_labels": torch.tensor([r.binary_label for r in records]),
            "original_labels": [r.original_label for r in records],
            "paths": [str(r.path) for r in records],
        },
        output_path,
    )
    print(f"Đã lưu {len(records)} embedding vào {output_path}")


In [64]:
# OVERVIEW: Lưu checkpoint model cùng cấu hình dataset/nhãn để tái lập thí nghiệm unlearning.
def save_checkpoint(
    path: Path,
    model: FlowModel,
    embedding_dim: int,
    hidden_dims: Sequence[int],
    dropout: float,
    max_packets: int,
    known_labels: set[str],
    unknown_labels: set[str],
    holdout_unknown_labels: set[str],
    unknown_threshold: float | None = None,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "model_config": {
                "embedding_dim": embedding_dim,
                "hidden_dims": list(hidden_dims),
                "dropout": dropout,
                "max_packets": max_packets,
            },
            "known_labels": sorted(known_labels),
            "unknown_labels": sorted(unknown_labels),
            "holdout_unknown_labels": sorted(holdout_unknown_labels),
            "unknown_threshold": unknown_threshold,
        },
        path,
    )
    print(f"Đã lưu model vào {path}")


In [65]:
# OVERVIEW: Nạp checkpoint đã train để đánh giá lại hoặc chạy instance-wise unlearning.
def load_checkpoint(
    path: Path,
    device: torch.device,
) -> tuple[FlowModel, dict[str, object]]:
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    config = checkpoint["model_config"]
    model = FlowModel(
        embedding_dim=int(config["embedding_dim"]),
        hidden_dims=tuple(int(value) for value in config["hidden_dims"]),
        dropout=float(config["dropout"]),
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    return model, checkpoint


In [66]:
# OVERVIEW: Chọn Df theo file hoặc nhãn; Df là instance cần quên, Dr là phần còn lại cần giữ.
def select_forget_records(
    records: Sequence[FlowRecord],
    root: Path,
    forget_list: Path | None,
    forget_labels: set[str],
    forget_fraction: float,
    seed: int,
) -> tuple[list[FlowRecord], list[FlowRecord]]:
    if forget_list:
        requested: set[Path] = set()
        requested_names: set[str] = set()
        for line in forget_list.read_text(encoding="utf-8").splitlines():
            value = line.strip()
            if not value or value.startswith("#"):
                continue
            candidate = Path(value)
            resolved = (candidate if candidate.is_absolute() else root / candidate).resolve()
            requested.add(resolved)
            requested_names.add(candidate.name)
        forget = [
            record
            for record in records
            if record.path.resolve() in requested or record.path.name in requested_names
        ]
    elif forget_labels:
        candidates = [record for record in records if record.original_label in forget_labels]
        rng = random.Random(seed)
        rng.shuffle(candidates)
        count = max(1, int(len(candidates) * forget_fraction))
        forget = candidates[:count]
    else:
        raise ValueError("Cần chỉ định --forget-list hoặc --forget-labels")

    forget_paths = {record.path.resolve() for record in forget}
    retain = [record for record in records if record.path.resolve() not in forget_paths]
    if not forget:
        raise ValueError("Không tìm thấy mẫu nào trong forget set")
    if not retain:
        raise ValueError("Retain set không được rỗng")
    return retain, forget


In [67]:
# OVERVIEW: Lưu metrics/history ra JSON để so sánh baseline, open-world và unlearning.
def write_json(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


## Thực nghiệm
Các cell dưới đây dùng trực tiếp implementation ở phía trên.


In [68]:
# OVERVIEW: Quét toàn bộ nhãn gốc trong DATA_DIR trước khi lọc, lưu label_inventory.json để biết dataset có bao nhiêu class.
all_files = scan_flow_files(DATA_DIR)
all_label_counts = Counter(label for _, label in all_files)
LABEL_INVENTORY_PATH = OUTPUT_DIR / 'label_inventory.json'

label_inventory = {
    'data_dirs': [str(path) for path in DATA_DIR],
    'total_files': len(all_files),
    'num_labels': len(all_label_counts),
    'min_samples_per_label_for_split': MIN_SAMPLES_PER_LABEL,
    'labels': [
        {'label': label, 'count': count}
        for label, count in sorted(all_label_counts.items())
    ],
}
write_json(LABEL_INVENTORY_PATH, label_inventory)

eligible_labels = [
    label for label, count in all_label_counts.items()
    if count >= MIN_SAMPLES_PER_LABEL
]
small_labels = [
    (label, count) for label, count in sorted(all_label_counts.items())
    if count < MIN_SAMPLES_PER_LABEL
]

print(f'Tổng số file PCAP/CAP/TXT/CSV: {len(all_files)}')
print(f'Tổng số nhãn gốc: {len(all_label_counts)}')
print(f'Nhãn đủ điều kiện split >= {MIN_SAMPLES_PER_LABEL} samples: {len(eligible_labels)}')
print(f'Nhãn bị loại vì quá ít sample: {len(small_labels)}')
print(f'Đã lưu inventory: {LABEL_INVENTORY_PATH}')
print('Top nhãn theo số sample:')
for label, count in sorted(all_label_counts.items(), key=lambda item: (-item[1], item[0]))[:20]:
    print(f'  {label:35s} {count:6d}')
if small_labels:
    print('Các nhãn ít sample:')
    for label, count in small_labels[:20]:
        print(f'  {label:35s} {count:6d}')


Tổng số file PCAP/CAP/TXT/CSV: 9005
Tổng số nhãn gốc: 45
Nhãn đủ điều kiện split >= 100 samples: 45
Nhãn bị loại vì quá ít sample: 0
Đã lưu inventory: /content/drive/MyDrive/unlearning-artifacts/notebook/label_inventory.json
Top nhãn theo số sample:
  united_airlines                        205
  37_200_000                             200
  english_to_spanish                     200
  food_near_me                           200
  office_365                             200
  office_depot                           200
  offset                                 200
  ohio_state_football                    200
  omegle                                 200
  paul_pelosi                            200
  phillies_game                          200
  powerball_jackpot                      200
  qr_code_generator                      200
  quavo                                  200
  quest_diagnostics                      200
  quickbooks                             200
  quizlet                     

In [69]:
# OVERVIEW: Random split nhãn ở cấp class vào known/unknown-train/holdout, lưu label_split theo seed, số nhãn và ratio để tái lập thí nghiệm.
split_size_name = 'all' if MAX_LABELS_FOR_EXPERIMENT is None else str(MAX_LABELS_FOR_EXPERIMENT)
known_ratio_name = int(KNOWN_LABEL_RATIO * 100)
unknown_ratio_name = int(UNKNOWN_TRAIN_LABEL_RATIO * 100)
LABEL_SPLIT_PATH = OUTPUT_DIR / f'label_split_seed{LABEL_SPLIT_SEED}_labels{split_size_name}_k{known_ratio_name}_u{unknown_ratio_name}.json'

if AUTO_LABEL_SPLIT and LABEL_SPLIT_PATH.exists() and not REBUILD_LABEL_SPLIT:
    split_payload = json.loads(LABEL_SPLIT_PATH.read_text(encoding='utf-8'))
    KNOWN_LABELS = set(split_payload['known_labels'])
    UNKNOWN_LABELS = set(split_payload['unknown_train_labels'])
    HOLDOUT_UNKNOWN_LABELS = set(split_payload['holdout_unknown_labels'])
    print(f'Đọc label split có sẵn: {LABEL_SPLIT_PATH}')
elif AUTO_LABEL_SPLIT:
    candidate_labels = sorted(
        label for label, count in all_label_counts.items()
        if count >= MIN_SAMPLES_PER_LABEL
    )
    if len(candidate_labels) < 3:
        raise ValueError(
            f'Cần ít nhất 3 nhãn đủ điều kiện để chia known/unknown/holdout, hiện có {len(candidate_labels)}.'
        )

    rng = random.Random(LABEL_SPLIT_SEED)
    rng.shuffle(candidate_labels)
    if MAX_LABELS_FOR_EXPERIMENT is not None:
        if MAX_LABELS_FOR_EXPERIMENT < 3:
            raise ValueError('MAX_LABELS_FOR_EXPERIMENT phải >= 3 hoặc bằng None.')
        candidate_labels = candidate_labels[:MAX_LABELS_FOR_EXPERIMENT]

    total_labels = len(candidate_labels)
    n_known = max(1, round(total_labels * KNOWN_LABEL_RATIO))
    n_unknown_train = max(1, round(total_labels * UNKNOWN_TRAIN_LABEL_RATIO))
    if n_known + n_unknown_train >= total_labels:
        n_known = max(1, total_labels - 2)
        n_unknown_train = 1
    n_holdout = total_labels - n_known - n_unknown_train
    if n_holdout < 1:
        raise ValueError('Không còn nhãn cho holdout unknown; hãy giảm KNOWN_LABEL_RATIO hoặc UNKNOWN_TRAIN_LABEL_RATIO.')

    KNOWN_LABELS = set(candidate_labels[:n_known])
    UNKNOWN_LABELS = set(candidate_labels[n_known:n_known + n_unknown_train])
    HOLDOUT_UNKNOWN_LABELS = set(candidate_labels[n_known + n_unknown_train:])

    split_payload = {
        'seed': LABEL_SPLIT_SEED,
        'min_samples_per_label': MIN_SAMPLES_PER_LABEL,
        'max_labels_for_experiment': MAX_LABELS_FOR_EXPERIMENT,
        'known_label_ratio': KNOWN_LABEL_RATIO,
        'unknown_train_label_ratio': UNKNOWN_TRAIN_LABEL_RATIO,
        'holdout_unknown_label_ratio': 1.0 - KNOWN_LABEL_RATIO - UNKNOWN_TRAIN_LABEL_RATIO,
        'known_labels': sorted(KNOWN_LABELS),
        'unknown_train_labels': sorted(UNKNOWN_LABELS),
        'holdout_unknown_labels': sorted(HOLDOUT_UNKNOWN_LABELS),
        'label_counts': {label: all_label_counts[label] for label in sorted(candidate_labels)},
    }
    write_json(LABEL_SPLIT_PATH, split_payload)
    print(f'Tạo label split mới: {LABEL_SPLIT_PATH}')
else:
    split_payload = {
        'mode': 'manual',
        'known_labels': sorted(KNOWN_LABELS),
        'unknown_train_labels': sorted(UNKNOWN_LABELS),
        'holdout_unknown_labels': sorted(HOLDOUT_UNKNOWN_LABELS),
    }
    print('AUTO_LABEL_SPLIT=False, dùng KNOWN_LABELS/UNKNOWN_LABELS/HOLDOUT_UNKNOWN_LABELS thủ công trong cell cấu hình.')

selected_labels = KNOWN_LABELS | UNKNOWN_LABELS | HOLDOUT_UNKNOWN_LABELS
missing_labels = sorted(label for label in selected_labels if label not in all_label_counts)
if missing_labels:
    raise ValueError(f'Label split chứa nhãn không tồn tại trong DATA_DIR: {missing_labels}')

def print_label_group(title: str, labels: set[str]) -> None:
    total_samples = sum(all_label_counts[label] for label in labels)
    print(f'{title}: {len(labels)} nhãn, {total_samples} samples')
    for label in sorted(labels):
        print(f'  {label:35s} {all_label_counts[label]:6d}')

print('Tóm tắt label split:')
print_label_group('KNOWN_LABELS', KNOWN_LABELS)
print_label_group('UNKNOWN_LABELS train', UNKNOWN_LABELS)
print_label_group('HOLDOUT_UNKNOWN_LABELS test-only', HOLDOUT_UNKNOWN_LABELS)
print(f'Tổng nhãn được dùng: {len(selected_labels)} / {len(all_label_counts)}')
print(f'Tổng sample được dùng: {sum(all_label_counts[label] for label in selected_labels)} / {len(all_files)}')
print(f'Đường dẫn split: {LABEL_SPLIT_PATH}')


Tạo label split mới: /content/drive/MyDrive/unlearning-artifacts/notebook/label_split_seed42_labels24_k45_u35.json
Tóm tắt label split:
KNOWN_LABELS: 11 nhãn, 2205 samples
  office_365                             200
  phillies_game                          200
  powerball_jackpot                      200
  quavo                                  200
  realtor                                200
  soap2day                               200
  solitaire                              200
  ticketmaster                           200
  united_airlines                        205
  united_states_elections_2022           200
  us_house_elections_2022                200
UNKNOWN_LABELS train: 8 nhãn, 1600 samples
  office_depot                           200
  qr_code_generator                      200
  take_off_dead                          200
  taylor_swift                           200
  twitter                                200
  usps                                   200
  v_for_vendetta    

In [70]:
# OVERVIEW: Tạo FlowRecord sau label split và giới hạn số PCAP mỗi nhãn để quick debug không phải parse quá nhiều flow.
if not all(path.exists() and path.is_dir() for path in DATA_DIR):
    raise FileNotFoundError(f'Không tìm thấy một trong các DATA_DIR: {DATA_DIR}')

records = build_records(
    DATA_DIR,
    KNOWN_LABELS,
    UNKNOWN_LABELS,
    HOLDOUT_UNKNOWN_LABELS,
)

if MAX_FILES_PER_LABEL is not None:
    before_cap = len(records)
    rng = random.Random(SEED)
    grouped_records: dict[str, list[FlowRecord]] = {}
    for record in records:
        grouped_records.setdefault(record.original_label, []).append(record)

    capped_records: list[FlowRecord] = []
    for label in sorted(grouped_records):
        group = grouped_records[label]
        if len(group) > MAX_FILES_PER_LABEL:
            rng.shuffle(group)
            group = group[:MAX_FILES_PER_LABEL]
        capped_records.extend(group)
    records = sorted(capped_records, key=lambda record: str(record.path))
    print(f'Giới hạn mỗi nhãn tối đa {MAX_FILES_PER_LABEL} PCAP: {before_cap} -> {len(records)} samples')
else:
    print('Không giới hạn số PCAP mỗi nhãn; dùng toàn bộ sample thuộc label split.')

label_counts = Counter(record.original_label for record in records)
role_counts = Counter(
    'known' if record.original_label in KNOWN_LABELS
    else ('unknown-holdout' if record.original_label in HOLDOUT_UNKNOWN_LABELS else 'unknown-train')
    for record in records
)

print(f'Tổng số sample sau khi lọc theo label split: {len(records)}')
print('Số sample theo role:', dict(sorted(role_counts.items())))
print('Chi tiết nhãn được dùng:')
for label, count in sorted(label_counts.items()):
    role = 'known' if label in KNOWN_LABELS else ('unknown-holdout' if label in HOLDOUT_UNKNOWN_LABELS else 'unknown-train')
    print(f'{label:35s} {count:6d}  {role}')


Giới hạn mỗi nhãn tối đa 160 PCAP: 4805 -> 3840 samples
Tổng số sample sau khi lọc theo label split: 3840
Số sample theo role: {'known': 1760, 'unknown-holdout': 800, 'unknown-train': 1280}
Chi tiết nhãn được dùng:
office_365                             160  known
office_depot                           160  unknown-train
phillies_game                          160  known
powerball_jackpot                      160  known
qr_code_generator                      160  unknown-train
quavo                                  160  known
quest_diagnostics                      160  unknown-holdout
realtor                                160  known
ross                                   160  unknown-holdout
soap2day                               160  known
social_security                        160  unknown-holdout
solitaire                              160  known
take_off_dead                          160  unknown-train
takeoff_shooting                       160  unknown-holdout
taylor_swift         

In [71]:
# OVERVIEW: Split dữ liệu và tạo DataLoader lazy; validation gồm cả unknown-train và holdout-unknown để tune threshold.
train_records, val_records, test_records = split_records(
    records,
    HOLDOUT_UNKNOWN_LABELS,
    SEED,
    holdout_validation_ratio=HOLDOUT_VALIDATION_RATIO,
)

PACKET_CACHE_DIR = OUTPUT_DIR / 'packet-feature-cache'
FEATURE_CACHE_DIR = PACKET_CACHE_DIR / 'features'
train_loader = make_loader(train_records, MAX_PACKETS, BATCH_SIZE, shuffle=True, cache_dir=FEATURE_CACHE_DIR)
val_loader = make_loader(val_records, MAX_PACKETS, BATCH_SIZE, shuffle=False, cache_dir=FEATURE_CACHE_DIR)
test_loader = make_loader(test_records, MAX_PACKETS, BATCH_SIZE, shuffle=False, cache_dir=FEATURE_CACHE_DIR)

def binary_counts(records_subset: Sequence[FlowRecord]) -> dict[str, int]:
    counts = Counter(record.binary_label for record in records_subset)
    return {'known': counts.get(0, 0), 'unknown': counts.get(1, 0)}

print(f'train={len(train_records)}, validation={len(val_records)}, test={len(test_records)}')
print('Binary counts:')
print('  train     :', binary_counts(train_records))
print('  validation:', binary_counts(val_records))
print('  test      :', binary_counts(test_records))
print(f'Packet feature cache: {FEATURE_CACHE_DIR}')
features, mask, label = train_loader.dataset[0]
print(f'Packet tensor: {tuple(features.shape)}')
print(f'Mask tensor: {tuple(mask.shape)}, label: {label.item()}')


train=2128, validation=656, test=1056
Binary counts:
  train     : {'known': 1232, 'unknown': 896}
  validation: {'known': 264, 'unknown': 392}
  test      : {'known': 264, 'unknown': 792}
Packet feature cache: /content/drive/MyDrive/unlearning-artifacts/notebook/packet-feature-cache/features
Packet tensor: (256, 5)
Mask tensor: (256,), label: 1


In [72]:
# OVERVIEW: Khởi tạo mô hình đúng pipeline trong plan: encoder 256 chiều + MLP output known/unknown.
if 'DEVICE' not in globals():
    DEVICE = select_device(DEVICE_NAME, REQUIRE_CUDA)
    print(f'Device đang dùng: {DEVICE}')

model = FlowModel(
    embedding_dim=EMBEDDING_DIM,
    hidden_dims=HIDDEN_DIMS,
    dropout=DROPOUT,
).to(DEVICE)

number_of_parameters = sum(parameter.numel() for parameter in model.parameters())
print(model)
print(f'Tổng số tham số: {number_of_parameters:,}')


FlowModel(
  (encoder): FlowEncoder(
    (network): Sequential(
      (0): Conv1d(5, 64, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
      (4): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): GELU(approximate='none')
    )
    (projection): Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
  )
  (classifier): MLPClassifier(
    (network): Sequential(
      (0): Linear(in_features=256, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, e

In [73]:
# OVERVIEW: Train end-to-end encoder và MLP trên known + unknown-train; dùng class weights và balanced accuracy để giảm lệch về known.
training_result = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    max_train_batches_per_epoch=MAX_TRAIN_BATCHES_PER_EPOCH,
    max_eval_batches=MAX_EVAL_BATCHES,
    log_every_n_batches=LOG_EVERY_N_BATCHES,
    use_class_weights=USE_CLASS_WEIGHTS,
    checkpoint_score_metric=CHECKPOINT_SCORE_METRIC,
    unknown_threshold=UNKNOWN_THRESHOLD,
)
print(f'Best validation score ({CHECKPOINT_SCORE_METRIC}): {training_result["best_score"]:.4f}')


Train label counts=[1232, 896], class_weights=[0.8421052098274231, 1.1578946113586426]
Train config: epochs=12, batch_size=128, train_batches=17, val_batches=6, max_train_batches_per_epoch=None, max_eval_batches=None, log_every_n_batches=2, use_class_weights=True, checkpoint_score_metric=balanced_accuracy, unknown_threshold=0.5


/usr/local/lib/python3.13/dist-packages/scapy/layers/tls/crypto/groups.py:25: CryptographyDeprecationWarning: Diffie-Hellman over finite fields (FFDH) is deprecated and support will be removed in a future release. Use a more modern key exchange algorithm.
  from cryptography.hazmat.primitives.asymmetric.dh import DHParameterNumbers


Epoch 001/12 | batch 0001/17 | loss=0.7522 | elapsed=35.9s
Epoch 001/12 | batch 0002/17 | loss=0.7008 | elapsed=71.1s
Epoch 001/12 | batch 0004/17 | loss=0.7065 | elapsed=150.2s
Epoch 001/12 | batch 0006/17 | loss=0.7042 | elapsed=223.6s
Epoch 001/12 | batch 0008/17 | loss=0.7032 | elapsed=293.0s
Epoch 001/12 | batch 0010/17 | loss=0.6739 | elapsed=351.7s
Epoch 001/12 | batch 0012/17 | loss=0.7208 | elapsed=424.7s
Epoch 001/12 | batch 0014/17 | loss=0.6862 | elapsed=492.8s
Epoch 001/12 | batch 0016/17 | loss=0.6805 | elapsed=567.2s
Epoch 001/12 done | train_loss=0.7008 | val_acc=0.5976 | val_known_recall=0.0000 | val_unknown_recall=1.0000 | val_balanced_acc=0.5000 | score=0.5000 | epoch_time=902.0s | val_time=306.4s | confusion=[[0, 264], [0, 392]]
Epoch 002/12 | batch 0001/17 | loss=0.7140 | elapsed=0.3s
Epoch 002/12 | batch 0002/17 | loss=0.6977 | elapsed=0.6s
Epoch 002/12 | batch 0004/17 | loss=0.6765 | elapsed=1.1s
Epoch 002/12 | batch 0006/17 | loss=0.6697 | elapsed=1.6s
Epoch 002

In [74]:
# OVERVIEW: Sweep threshold unknown trên validation để chọn ngưỡng tốt nhất theo balanced accuracy trước khi test.
def sweep_unknown_threshold(
    model: FlowModel,
    loader: DataLoader,
    device: torch.device,
    thresholds: Sequence[float],
) -> dict[str, object]:
    rows: list[dict[str, object]] = []
    for threshold in thresholds:
        metrics = evaluate(model, loader, device, unknown_threshold=float(threshold))
        row = {
            'threshold': float(threshold),
            'accuracy': float(metrics['accuracy']),
            'known_recall': float(metrics['known_recall']),
            'unknown_recall': float(metrics['unknown_recall']),
            'unknown_precision': float(metrics['unknown_precision']),
            'balanced_accuracy': float(metrics['balanced_accuracy']),
            'confusion_matrix': metrics['confusion_matrix'],
        }
        rows.append(row)

    best = max(
        rows,
        key=lambda row: (
            row['balanced_accuracy'],
            row['unknown_recall'],
            row['accuracy'],
        ),
    )
    return {'best': best, 'rows': rows}

if USE_THRESHOLD_SWEEP:
    threshold_result = sweep_unknown_threshold(model, val_loader, DEVICE, THRESHOLD_GRID)
    BEST_UNKNOWN_THRESHOLD = float(threshold_result['best']['threshold'])
    print(f'Best unknown threshold: {BEST_UNKNOWN_THRESHOLD:.2f}')
    print('Threshold sweep trên validation:')
    for row in threshold_result['rows']:
        print(
            f"thr={row['threshold']:.2f} | bal_acc={row['balanced_accuracy']:.4f} | "
            f"known_recall={row['known_recall']:.4f} | unknown_recall={row['unknown_recall']:.4f} | "
            f"unknown_precision={row['unknown_precision']:.4f} | confusion={row['confusion_matrix']}"
        )
else:
    threshold_result = {'best': {'threshold': UNKNOWN_THRESHOLD}, 'rows': []}
    BEST_UNKNOWN_THRESHOLD = UNKNOWN_THRESHOLD
    print(f'Không sweep threshold; dùng UNKNOWN_THRESHOLD={BEST_UNKNOWN_THRESHOLD:.2f}')


Best unknown threshold: 0.45
Threshold sweep trên validation:
thr=0.20 | bal_acc=0.5737 | known_recall=0.3106 | unknown_recall=0.8367 | unknown_precision=0.6431 | confusion=[[82, 182], [64, 328]]
thr=0.25 | bal_acc=0.5862 | known_recall=0.3485 | unknown_recall=0.8240 | unknown_precision=0.6525 | confusion=[[92, 172], [69, 323]]
thr=0.30 | bal_acc=0.5836 | known_recall=0.3636 | unknown_recall=0.8036 | unknown_precision=0.6522 | confusion=[[96, 168], [77, 315]]
thr=0.35 | bal_acc=0.5993 | known_recall=0.4205 | unknown_recall=0.7781 | unknown_precision=0.6659 | confusion=[[111, 153], [87, 305]]
thr=0.40 | bal_acc=0.6283 | known_recall=0.5985 | unknown_recall=0.6582 | unknown_precision=0.7088 | confusion=[[158, 106], [134, 258]]
thr=0.45 | bal_acc=0.6583 | known_recall=0.7197 | unknown_recall=0.5969 | unknown_precision=0.7597 | confusion=[[190, 74], [158, 234]]
thr=0.50 | bal_acc=0.6529 | known_recall=0.7727 | unknown_recall=0.5332 | unknown_precision=0.7770 | confusion=[[204, 60], [183, 2

In [75]:
# OVERVIEW: Đánh giá test bằng threshold đã tune, in accuracy/balanced accuracy/confusion matrix rồi lưu checkpoint.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_UNKNOWN_THRESHOLD = globals().get('BEST_UNKNOWN_THRESHOLD', UNKNOWN_THRESHOLD)
test_metrics = evaluate(model, test_loader, DEVICE, max_batches=MAX_TEST_BATCHES, unknown_threshold=BEST_UNKNOWN_THRESHOLD)
print('Test metrics:', test_metrics)
print('Confusion matrix format: [[known->known, known->unknown], [unknown->known, unknown->unknown]]')
print(f'BEST_UNKNOWN_THRESHOLD={BEST_UNKNOWN_THRESHOLD:.2f}; MAX_TEST_BATCHES={MAX_TEST_BATCHES}; đặt None nếu cần evaluate toàn bộ test set.')

save_checkpoint(
    OUTPUT_DIR / 'best_model.pt',
    model,
    EMBEDDING_DIM,
    HIDDEN_DIMS,
    DROPOUT,
    MAX_PACKETS,
    KNOWN_LABELS,
    UNKNOWN_LABELS,
    HOLDOUT_UNKNOWN_LABELS,
    unknown_threshold=BEST_UNKNOWN_THRESHOLD,
)
if SAVE_EMBEDDINGS_AFTER_TRAIN:
    embedding_records = records if MAX_EMBEDDING_RECORDS is None else records[:MAX_EMBEDDING_RECORDS]
    print(f'Lưu embedding cho {len(embedding_records)} / {len(records)} records.')
    save_embeddings(
        model,
        embedding_records,
        OUTPUT_DIR / 'embeddings.pt',
        MAX_PACKETS,
        BATCH_SIZE,
        DEVICE,
        cache_dir=FEATURE_CACHE_DIR,
    )
else:
    print('Bỏ qua lưu embeddings.pt để cell test chạy nhanh. Đặt SAVE_EMBEDDINGS_AFTER_TRAIN=True nếu cần.')


Test metrics: {'loss': 0.8527102036909624, 'accuracy': 0.5691287878787878, 'known_recall': 0.6931818181818182, 'unknown_recall': 0.5277777777777778, 'unknown_precision': 0.8376753507014028, 'balanced_accuracy': 0.610479797979798, 'unknown_threshold': 0.45, 'known_total': 264, 'unknown_total': 792, 'predicted_unknown_total': 499, 'confusion_matrix': [[183, 81], [374, 418]]}
Confusion matrix format: [[known->known, known->unknown], [unknown->known, unknown->unknown]]
BEST_UNKNOWN_THRESHOLD=0.45; MAX_TEST_BATCHES=None; đặt None nếu cần evaluate toàn bộ test set.
Đã lưu model vào /content/drive/MyDrive/unlearning-artifacts/notebook/best_model.pt
Bỏ qua lưu embeddings.pt để cell test chạy nhanh. Đặt SAVE_EMBEDDINGS_AFTER_TRAIN=True nếu cần.


In [76]:
# OVERVIEW: Smoke test 5 PCAP từ test set bằng threshold đã tune: dự đoán và đối chiếu đúng/sai.
SMOKE_TEST_N = 5
BEST_UNKNOWN_THRESHOLD = globals().get('BEST_UNKNOWN_THRESHOLD', UNKNOWN_THRESHOLD)
rng = random.Random(SEED)
known_test_records = [record for record in test_records if record.binary_label == 0]
unknown_test_records = [record for record in test_records if record.binary_label == 1]

target_unknown = min(len(unknown_test_records), max(1, SMOKE_TEST_N // 2 + SMOKE_TEST_N % 2))
target_known = min(len(known_test_records), SMOKE_TEST_N - target_unknown)
smoke_records = []
if target_known:
    smoke_records.extend(rng.sample(known_test_records, target_known))
if target_unknown:
    smoke_records.extend(rng.sample(unknown_test_records, target_unknown))
if len(smoke_records) < SMOKE_TEST_N:
    selected_paths = {record.path for record in smoke_records}
    remaining_records = [record for record in test_records if record.path not in selected_paths]
    smoke_records.extend(rng.sample(remaining_records, min(SMOKE_TEST_N - len(smoke_records), len(remaining_records))))
rng.shuffle(smoke_records)

model.eval()
correct = 0
print(f'Smoke test trên {len(smoke_records)} mẫu từ test_records, threshold={BEST_UNKNOWN_THRESHOLD:.2f}')
print('Format: OK? | true_binary | pred_binary | p_unknown | confidence | original_label | path')
for index, sample_record in enumerate(smoke_records, start=1):
    sample_dataset = FlowDataset([sample_record], MAX_PACKETS, cache_dir=FEATURE_CACHE_DIR)
    sample_features, sample_mask, sample_label = sample_dataset[0]
    with torch.no_grad():
        logits, embedding = model(
            sample_features.unsqueeze(0).to(DEVICE),
            sample_mask.unsqueeze(0).to(DEVICE),
        )
        probabilities = torch.softmax(logits, dim=1).squeeze(0).detach().cpu()

    p_unknown = float(probabilities[1].item())
    prediction = int(p_unknown >= BEST_UNKNOWN_THRESHOLD)
    true_label = int(sample_label.item())
    is_correct = prediction == true_label
    correct += int(is_correct)
    true_name = 'unknown' if true_label == 1 else 'known'
    pred_name = 'unknown' if prediction == 1 else 'known'
    confidence = p_unknown if prediction == 1 else 1.0 - p_unknown
    status = 'OK' if is_correct else 'WRONG'
    print(
        f'[{index}] {status:5s} | true={true_name:7s} | pred={pred_name:7s} | '
        f'p_unknown={p_unknown:.4f} | conf={confidence:.4f} | '
        f'original={sample_record.original_label} | path={sample_record.path.name}'
    )

smoke_accuracy = correct / max(len(smoke_records), 1)
print(f'Smoke accuracy: {correct}/{len(smoke_records)} = {smoke_accuracy:.4f}')
print('Lưu ý: smoke test 5 mẫu chỉ để kiểm tra trực quan; metric chính vẫn là Test metrics ở cell trước.')


Smoke test trên 5 mẫu từ test_records, threshold=0.45
Format: OK? | true_binary | pred_binary | p_unknown | confidence | original_label | path
[1] OK    | true=unknown | pred=unknown | p_unknown=0.8093 | conf=0.8093 | original=xfinity_customer_service | path=xfinity_customer_service_40.pcap
[2] OK    | true=known   | pred=known   | p_unknown=0.3747 | conf=0.6253 | original=solitaire | path=solitaire_70.pcap
[3] OK    | true=unknown | pred=unknown | p_unknown=0.8801 | conf=0.8801 | original=social_security | path=social_security_104.pcap
[4] OK    | true=unknown | pred=unknown | p_unknown=0.6798 | conf=0.6798 | original=qr_code_generator | path=qr_code_generator_34.pcap
[5] OK    | true=known   | pred=known   | p_unknown=0.3529 | conf=0.6471 | original=soap2day | path=soap2day_66.pcap
Smoke accuracy: 5/5 = 1.0000
Lưu ý: smoke test 5 mẫu chỉ để kiểm tra trực quan; metric chính vẫn là Test metrics ở cell trước.


In [77]:
# OVERVIEW: Chạy thử instance-wise unlearning theo plan: quên Df và giữ Dr bằng retain loss + regularization.
RUN_UNLEARNING = False
FORGET_LIST = Path('./forget.txt')
UNLEARN_OUTPUT_DIR = OUTPUT_DIR / 'unlearned'

if RUN_UNLEARNING:
    if not FORGET_LIST.exists():
        raise FileNotFoundError(f'Không tìm thấy forget list: {FORGET_LIST}')

    retain_records, forget_records = select_forget_records(
        train_records,
        DATA_DIR,
        FORGET_LIST,
        set(),
        1.0,
        SEED,
    )
    retain_loader = make_loader(retain_records, MAX_PACKETS, BATCH_SIZE, shuffle=True, cache_dir=PACKET_CACHE_DIR / 'retain')
    forget_loader = make_loader(forget_records, MAX_PACKETS, BATCH_SIZE, shuffle=True, cache_dir=PACKET_CACHE_DIR / 'forget')
    print(f'Retain={len(retain_records)}, Forget={len(forget_records)}')
    print('Before forget:', evaluate(model, forget_loader, DEVICE))
    print('Before retain:', evaluate(model, retain_loader, DEVICE))

    unlearning_history = run_unlearning(
        model,
        retain_loader,
        forget_loader,
        DEVICE,
        epochs=10,
        learning_rate=1e-4,
        retain_weight=1.0,
        regularization_weight=1e-3,
    )
    print('After forget:', evaluate(model, forget_loader, DEVICE))
    print('After retain:', evaluate(model, retain_loader, DEVICE))
else:
    print('Bỏ qua unlearning. Đặt RUN_UNLEARNING = True để chạy cell này.')

Bỏ qua unlearning. Đặt RUN_UNLEARNING = True để chạy cell này.


In [78]:
# OVERVIEW: Tổng hợp runtime, dataset split, train metrics, test metrics và smoke test vào run_summary.json/md cho người và AI đọc.
def to_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, set):
        return sorted(value)
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if hasattr(value, 'item'):
        return value.item()
    return value


def count_by_binary(records_subset: Sequence[FlowRecord]) -> dict[str, int]:
    counts = Counter(record.binary_label for record in records_subset)
    return {'known': counts.get(0, 0), 'unknown': counts.get(1, 0)}


def count_by_original_label(records_subset: Sequence[FlowRecord]) -> dict[str, int]:
    return dict(sorted(Counter(record.original_label for record in records_subset).items()))

summary = {
    'runtime': {
        'torch_version': torch.__version__,
        'device': str(DEVICE),
        'cuda_available': bool(torch.cuda.is_available()),
        'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    'config': {
        'dataset_dirs': [str(path) for path in DATA_DIR],
        'max_packets': MAX_PACKETS,
        'embedding_dim': EMBEDDING_DIM,
        'hidden_dims': list(HIDDEN_DIMS),
        'dropout': DROPOUT,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'max_labels_for_experiment': MAX_LABELS_FOR_EXPERIMENT,
        'max_files_per_label': MAX_FILES_PER_LABEL,
        'use_class_weights': USE_CLASS_WEIGHTS,
        'checkpoint_score_metric': CHECKPOINT_SCORE_METRIC,
        'unknown_threshold': UNKNOWN_THRESHOLD,
        'best_unknown_threshold': globals().get('BEST_UNKNOWN_THRESHOLD', UNKNOWN_THRESHOLD),
        'use_threshold_sweep': USE_THRESHOLD_SWEEP,
        'threshold_grid': THRESHOLD_GRID,
        'holdout_validation_ratio': HOLDOUT_VALIDATION_RATIO,
        'max_train_batches_per_epoch': MAX_TRAIN_BATCHES_PER_EPOCH,
        'max_eval_batches': MAX_EVAL_BATCHES,
        'max_test_batches': MAX_TEST_BATCHES,
        'seed': SEED,
    },
    'labels': {
        'known_labels': sorted(KNOWN_LABELS),
        'unknown_train_labels': sorted(UNKNOWN_LABELS),
        'holdout_unknown_labels': sorted(HOLDOUT_UNKNOWN_LABELS),
    },
    'split': {
        'total_records': len(records),
        'train': len(train_records),
        'validation': len(val_records),
        'test': len(test_records),
        'train_binary_counts': count_by_binary(train_records),
        'validation_binary_counts': count_by_binary(val_records),
        'test_binary_counts': count_by_binary(test_records),
        'selected_original_label_counts': count_by_original_label(records),
    },
    'training': to_jsonable(training_result) if 'training_result' in globals() else None,
    'threshold_sweep': to_jsonable(threshold_result) if 'threshold_result' in globals() else None,
    'test_metrics': to_jsonable(test_metrics) if 'test_metrics' in globals() else None,
    'smoke_test': {
        'n': len(smoke_records) if 'smoke_records' in globals() else None,
        'accuracy': smoke_accuracy if 'smoke_accuracy' in globals() else None,
    },
}

summary_json_path = OUTPUT_DIR / 'run_summary.json'
summary_md_path = OUTPUT_DIR / 'run_summary.md'
write_json(summary_json_path, summary)

lines = [
    '# Run Summary - train_pipeline.ipynb',
    '',
    '## Runtime',
    f"- device: `{summary['runtime']['device']}`",
    f"- cuda_available: `{summary['runtime']['cuda_available']}`",
    f"- cuda_device: `{summary['runtime']['cuda_device']}`",
    f"- torch_version: `{summary['runtime']['torch_version']}`",
    '',
    '## Dataset And Split',
    f"- total_records: `{summary['split']['total_records']}`",
    f"- train: `{summary['split']['train']}` {summary['split']['train_binary_counts']}",
    f"- validation: `{summary['split']['validation']}` {summary['split']['validation_binary_counts']}",
    f"- test: `{summary['split']['test']}` {summary['split']['test_binary_counts']}",
    f"- known_labels: `{summary['labels']['known_labels']}`",
    f"- unknown_train_labels: `{summary['labels']['unknown_train_labels']}`",
    f"- holdout_unknown_labels: `{summary['labels']['holdout_unknown_labels']}`",
    '',
    '## Config',
    f"- max_labels_for_experiment: `{MAX_LABELS_FOR_EXPERIMENT}`",
    f"- max_files_per_label: `{MAX_FILES_PER_LABEL}`",
    f"- batch_size: `{BATCH_SIZE}`",
    f"- epochs: `{EPOCHS}`",
    f"- learning_rate: `{LEARNING_RATE}`",
    f"- checkpoint_score_metric: `{CHECKPOINT_SCORE_METRIC}`",
    f"- best_unknown_threshold: `{summary['config']['best_unknown_threshold']}`",
    '',
    '## Training',
    f"- best_score: `{summary['training']['best_score'] if summary['training'] else None}`",
    f"- total_seconds: `{summary['training']['total_seconds'] if summary['training'] else None}`",
    '',
    '## Test Metrics',
]
if summary['test_metrics']:
    for key, value in summary['test_metrics'].items():
        lines.append(f'- {key}: `{value}`')
else:
    lines.append('- Chưa có `test_metrics`. Hãy chạy cell test trước.')
lines.extend([
    '',
    '## Smoke Test',
    f"- n: `{summary['smoke_test']['n']}`",
    f"- accuracy: `{summary['smoke_test']['accuracy']}`",
])
summary_md_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

print(f'Đã lưu summary JSON: {summary_json_path}')
print(f'Đã lưu summary Markdown: {summary_md_path}')
print('File này phù hợp để gửi/đọc lại nhanh thay vì đọc toàn bộ output trong .ipynb.')


Đã lưu summary JSON: /content/drive/MyDrive/unlearning-artifacts/notebook/run_summary.json
Đã lưu summary Markdown: /content/drive/MyDrive/unlearning-artifacts/notebook/run_summary.md
File này phù hợp để gửi/đọc lại nhanh thay vì đọc toàn bộ output trong .ipynb.
